# LeetCode 330 — Patching Array bằng Python (Hard)

Notebook này giảng nhập môn bài toán:

```python
class Solution:
    def minPatches(self, nums: List[int], n: int) -> int:

```

Tên bài dịch nôm na là **"Vá mảng"**: ta có sẵn một túi "đồng xu" `nums`, muốn trả được **mọi** số tiền từ `1` đến `n` (mỗi đồng xu dùng tối đa một lần), và được phép **bỏ thêm** đồng xu mới vào túi. Hỏi phải bỏ thêm **ít nhất** bao nhiêu đồng?

Đây là bài Hard "đáng tiền" vì code cuối cùng chỉ khoảng **10 dòng**, nhưng để tin nó đúng thì phải hiểu một nhận xét rất đẹp về **tổng tập con** (subset sum). Mục tiêu của bài học:

1. Hiểu **tổng tập con** là gì, tự tay liệt kê bằng `itertools.combinations`, rồi học cách xây **tập mọi tổng tạo được** bằng `set` mà không cần liệt kê từng tập con.
2. Viết **Cách 1 — brute force** "thử mọi bộ vá, số lần vá tăng dần" (dùng `combinations_with_replacement`). Cách này chậm khủng khiếp nhưng **chắc chắn đúng**, nên ta dùng nó làm **trọng tài** kiểm tra các cách sau.
3. Viết **Cách 2 — trực giác**: "thiếu số nào nhỏ nhất thì vá đúng số đó", vẫn dùng tập tổng. Cách này đúng nhưng tốn bộ nhớ `O(n)` — với `n` tới `2^31 − 1` thì không thể chạy.
4. Rút ra **nhận xét then chốt**: nếu đã tạo được mọi số trong `[1, miss)` thì thêm một số `x ≤ miss` sẽ mở rộng thành `[1, miss + x)`. Từ đó cả "tập tổng khổng lồ" được thay bằng **một biến duy nhất** `miss`.
5. Viết **Cách 3 — greedy (tham lam) với `miss`**: dùng `nums[i]` nếu `nums[i] ≤ miss`, ngược lại vá chính `miss`. Thời gian `O(m + log n)`, bộ nhớ `O(1)`. Trace kỹ với `nums = [1, 5, 10], n = 20`, và **chứng minh** vì sao vá đúng `miss` là tối ưu.
6. **Cách 4 — so sánh chiến lược vá**: vá `miss` đối đầu với "vá số 1", "vá số lớn nhất còn thiếu", "vá số nhỏ hơn `miss`"… kèm **phản ví dụ** cụ thể cho thấy vì sao chúng thua.
7. Đối chiếu chéo các cách trên dữ liệu ngẫu nhiên, tránh các lỗi kinh điển (`<` thay `<=`, `miss < n` thay `miss <= n`, quên dừng khi đã phủ đủ, nhầm công thức khi dùng biến `reach`) và nhận ra cùng pattern ở các bài anh em như *Maximum Number of Consecutive Values You Can Make* (1798), *Minimum Number of Coins to be Added* (2952).

## 1. Đề bài

Đề LeetCode (dịch sát nghĩa):

> Cho một mảng số nguyên **đã sắp xếp tăng dần** `nums` và một số nguyên `n`. Hãy **thêm / vá** (add/patch) các phần tử vào mảng sao cho **mọi** số trong đoạn `[1, n]` (tính cả hai đầu) đều có thể biểu diễn thành **tổng của một số phần tử** trong mảng.
>
> Trả về **số lần vá ít nhất** cần thiết.

"Tổng của một số phần tử" nghĩa là chọn ra một **tập con** các phần tử (mỗi vị trí dùng tối đa **một lần**) rồi cộng lại.

**Example 1**

```text
Input: nums = [1,3], n = 6
Output: 1
Giải thích:
Các cách chọn phần tử của nums là [1], [3], [1,3], cho ra các tổng: 1, 3, 4.
Nếu ta thêm/vá số 2 vào nums, các cách chọn là [1], [2], [3], [1,3], [2,3], [1,2,3].
Các tổng có thể là 1, 2, 3, 4, 5, 6, đã phủ hết đoạn [1, 6].
Vậy chỉ cần 1 lần vá.
```

**Example 2**

```text
Input: nums = [1,5,10], n = 20
Output: 2
Giải thích: Hai số vá có thể là [2, 4].
```

**Example 3**

```text
Input: nums = [1,2,2], n = 5
Output: 0
```

**Constraints**

- `1 <= nums.length <= 1000`
- `1 <= nums[i] <= 10^4`
- `nums` được sắp xếp theo thứ tự **tăng dần** (không giảm, có thể có số trùng như `[1,2,2]`).
- `1 <= n <= 2^31 - 1`

### Ký hiệu dùng suốt notebook

- `m = len(nums)` — số phần tử ban đầu.
- "**Tạo được** số `v`" = có một tập con của mảng (sau khi vá) có tổng đúng bằng `v`.
- "**Phủ** `[1, n]`" = tạo được **mọi** số `1, 2, …, n`.
- `[a, b)` là **nửa khoảng**: gồm `a, a+1, …, b−1`, **không** gồm `b` (giống `range(a, b)` của Python).
- `miss` (viết tắt của *missing*) = số **nhỏ nhất chưa tạo được**. Đây là nhân vật chính của bài.

### 1.1 Hiểu đề bằng bảng

Với Example 1 `nums = [1, 3]`, `n = 6`, ta kiểm tra từng số từ `1` đến `6`:

| Số cần tạo | Trước khi vá (`[1, 3]`) | Sau khi vá `2` (`[1, 2, 3]`) |
|---:|---|---|
| 1 | `1` ✔ | `1` ✔ |
| 2 | ✘ không tạo được | `2` ✔ |
| 3 | `3` ✔ | `3` ✔ |
| 4 | `1 + 3` ✔ | `1 + 3` ✔ |
| 5 | ✘ | `2 + 3` ✔ |
| 6 | ✘ | `1 + 2 + 3` ✔ |

Vài điều cần để ý ngay từ đầu:

- Số vá **tùy ý chọn** (miễn là số nguyên dương), không bắt buộc nằm trong khoảng nào. Đề chỉ hỏi **bao nhiêu** lần vá, không hỏi vá số nào. Ở Example 2 có thể có nhiều bộ vá tối ưu khác nhau; đề chỉ cần con số `2`.
- Phần tử **trùng** được tính như hai đồng xu khác nhau: `[1, 2, 2]` tạo được `4 = 2 + 2` và `5 = 1 + 2 + 2`. Đó là lý do Example 3 không cần vá.
- `n` có thể tới `2^31 − 1 ≈ 2,1 tỉ`. Một thuật toán phải "duyệt từng số từ 1 tới `n`" là **hết hy vọng**. Đây là gợi ý lớn rằng lời giải chính phải nhảy rất nhanh.

Cell dưới khai báo các ví dụ của đề, dùng suốt notebook.

In [1]:
from typing import List
from itertools import combinations, combinations_with_replacement
from math import comb
import random
import time

EXAMPLES = [
    ([1, 3], 6, 1),        # Example 1
    ([1, 5, 10], 20, 2),   # Example 2
    ([1, 2, 2], 5, 0),     # Example 3
]

for nums, n, expected in EXAMPLES:
    print(f"nums = {nums!s:12}  n = {n:3}  -> đáp án mong đợi {expected}")

nums = [1, 3]        n =   6  -> đáp án mong đợi 1
nums = [1, 5, 10]    n =  20  -> đáp án mong đợi 2
nums = [1, 2, 2]     n =   5  -> đáp án mong đợi 0


## 2. Nền tảng

### 2.1 Tổng tập con (subset sum) là gì?

Một **tập con** (subset) của mảng là một cách **chọn ra một số phần tử** (có thể không chọn gì, có thể chọn tất cả). Mảng có `m` phần tử thì có `2^m` cách chọn, vì mỗi phần tử có 2 khả năng: *lấy* hoặc *không lấy*.

**Tổng tập con** = tổng các phần tử được chọn. Tập con rỗng có tổng `0` — ta vẫn giữ số `0` này trong tính toán vì nó rất tiện (giống "không dùng đồng xu nào").

Module `itertools` có hàm `combinations(arr, k)`: sinh mọi cách chọn **đúng `k` phần tử** theo vị trí (không lặp, không quan tâm thứ tự). Cho `k` chạy từ `0` đến `m` là liệt kê đủ mọi tập con.

Ví dụ mini với `nums = [1, 3]`: có `2^2 = 4` tập con.

In [2]:
nums = [1, 3]
for k in range(len(nums) + 1):
    for chosen in combinations(nums, k):
        print(f"chọn {list(chosen)!s:8} -> tổng {sum(chosen)}")

print("số tập con =", 2 ** len(nums))

chọn []       -> tổng 0
chọn [1]      -> tổng 1
chọn [3]      -> tổng 3
chọn [1, 3]   -> tổng 4
số tập con = 4


Kết quả khớp đúng phần giải thích của Example 1: các tổng **dương** là `1, 3, 4`.

Liệt kê từng tập con có hai nhược điểm:

1. Số tập con là `2^m` — với `m = 30` đã hơn 1 tỉ.
2. Nhiều tập con cho **cùng** một tổng (ví dụ `[1, 2, 3]`: `{3}` và `{1, 2}` đều cho `3`), ta tốn công liệt kê thừa.

Ta chỉ cần biết **tổng nào tạo được**, không cần biết tạo bằng cách nào. Nên ta chuyển sang lưu một **tập hợp các tổng**.

### 2.2 Xây tập tổng bằng `set`, từng phần tử một

`set` (tập hợp) trong Python lưu các giá trị **không trùng nhau**, kiểm tra `v in s` nhanh (trung bình `O(1)`). Vài phép cần dùng:

- `{0}` — tập chỉ có số 0.
- `{s + x for s in sums}` — **set comprehension**: tạo tập mới bằng cách cộng `x` vào từng phần tử của `sums`.
- `a | b` — **hợp** (union) hai tập; `a |= b` là "thêm mọi phần tử của `b` vào `a`".

Ý tưởng: giả sử `sums` là mọi tổng tạo được bằng các phần tử **đã xét**. Khi xét thêm phần tử `x`, mỗi tổng cũ `s` có hai lựa chọn: *không lấy* `x` (vẫn là `s`) hoặc *lấy* `x` (thành `s + x`). Vậy:

```text
sums_mới = sums_cũ  ∪  { s + x  với mọi s trong sums_cũ }
```

Tính tay với `[1, 3, 2]`:

| Xét `x` | `sums` trước | thêm `s + x` | `sums` sau |
|---:|---|---|---|
| 1 | `{0}` | `{1}` | `{0, 1}` |
| 3 | `{0, 1}` | `{3, 4}` | `{0, 1, 3, 4}` |
| 2 | `{0, 1, 3, 4}` | `{2, 3, 5, 6}` | `{0, 1, 2, 3, 4, 5, 6}` |

Tham số `limit`: vì bài chỉ quan tâm các số `≤ n`, ta **bỏ** mọi tổng lớn hơn `limit` để tập không phình to vô ích.

In [3]:
def subset_sums(nums, limit=None, verbose=False):
    # Trả về set mọi tổng tập con (kể cả 0). Nếu có limit thì chỉ giữ tổng <= limit.
    sums = {0}
    for x in nums:
        new = {s + x for s in sums}
        if limit is not None:
            new = {v for v in new if v <= limit}
        sums |= new
        if verbose:
            print(f"xét x = {x}: sums = {sorted(sums)}")
    return sums


subset_sums([1, 3, 2], verbose=True)
print("[1, 5, 10] ->", sorted(subset_sums([1, 5, 10])))
print("[1, 5, 10], limit=12 ->", sorted(subset_sums([1, 5, 10], limit=12)))
print("[1, 2, 2] ->", sorted(subset_sums([1, 2, 2])))

xét x = 1: sums = [0, 1]
xét x = 3: sums = [0, 1, 3, 4]
xét x = 2: sums = [0, 1, 2, 3, 4, 5, 6]
[1, 5, 10] -> [0, 1, 5, 6, 10, 11, 15, 16]
[1, 5, 10], limit=12 -> [0, 1, 5, 6, 10, 11]
[1, 2, 2] -> [0, 1, 2, 3, 4, 5]


Kích thước `sums` không bao giờ vượt quá `limit + 1` (các số `0..limit`), nên mỗi phần tử tốn `O(limit)` thao tác. Đây chính là kỹ thuật **DP tổng tập con** quen thuộc, chỉ khác là lưu bằng `set` thay vì mảng `True/False`.

### 2.3 Kiểm tra "phủ `[1, n]`" và tìm số nhỏ nhất còn thiếu

Có tập tổng rồi, hai câu hỏi của bài trở nên dễ:

- `smallest_missing(nums, n)`: duyệt `v = 1, 2, …, n`, gặp số đầu tiên **không** nằm trong tập tổng thì trả về; nếu không thiếu số nào thì trả `None`.
- `covers(nums, n)`: phủ đủ khi và chỉ khi không thiếu số nào.

In [4]:
def smallest_missing(nums, n):
    # Số nhỏ nhất trong [1, n] KHÔNG tạo được; None nếu phủ đủ [1, n].
    sums = subset_sums(nums, limit=n)
    for v in range(1, n + 1):
        if v not in sums:
            return v
    return None


def covers(nums, n):
    return smallest_missing(nums, n) is None


for nums, n, _ in EXAMPLES:
    print(f"nums = {nums!s:12} n = {n:3}: số nhỏ nhất còn thiếu = {smallest_missing(nums, n)}, phủ đủ? {covers(nums, n)}")
print("thêm 2 vào [1, 3]:", covers([1, 3, 2], 6))

nums = [1, 3]       n =   6: số nhỏ nhất còn thiếu = 2, phủ đủ? False
nums = [1, 5, 10]   n =  20: số nhỏ nhất còn thiếu = 2, phủ đủ? False
nums = [1, 2, 2]    n =   5: số nhỏ nhất còn thiếu = None, phủ đủ? True
thêm 2 vào [1, 3]: True


### 2.4 Chọn bộ vá có lặp: `combinations_with_replacement`

Khi đi tìm đáp án bằng brute force, ta sẽ cần thử **mọi bộ `k` số vá**. Hai điểm cần cẩn thận:

- Số vá **được phép trùng nhau** (vá hai số `1` là hợp lệ), nên không dùng `combinations` (không lặp) mà dùng `combinations_with_replacement(values, k)` — chọn `k` phần tử, **cho phép lặp**, không quan tâm thứ tự.
- Thứ tự không quan trọng: vá `{2, 4}` hay `{4, 2}` là một. Hàm này chỉ sinh mỗi **tổ hợp** một lần, dạng không giảm `(a ≤ b ≤ …)`.

Số bộ sinh ra là `C(V + k − 1, k)` với `V` = số giá trị được phép. Ví dụ `V = 3` (giá trị `1, 2, 3`), `k = 2` → `C(4, 2) = 6` bộ.

In [5]:
bo = list(combinations_with_replacement([1, 2, 3], 2))
print(bo)
print("số bộ =", len(bo), "| công thức C(3+2-1, 2) =", comb(3 + 2 - 1, 2))

# Nhìn trước độ "nổ" của brute force: giá trị vá trong [1, V], chọn k số
for V, k in [(6, 1), (20, 2), (20, 3), (100, 4), (10**6, 3)]:
    print(f"V = {V:>7}, k = {k}: {comb(V + k - 1, k):,} bộ")

[(1, 1), (1, 2), (1, 3), (2, 2), (2, 3), (3, 3)]
số bộ = 6 | công thức C(3+2-1, 2) = 6
V =       6, k = 1: 6 bộ
V =      20, k = 2: 210 bộ
V =      20, k = 3: 1,540 bộ
V =     100, k = 4: 4,421,275 bộ
V = 1000000, k = 3: 166,667,166,667,000,000 bộ


### 2.5 Nửa khoảng `[1, miss)` và `range`

Trong lời giải chính, ta sẽ mô tả "những số đã tạo được" bằng **một con số** `miss` với ý nghĩa: *mọi số trong `[1, miss)` đều tạo được*. Cách viết nửa khoảng (mở bên phải) có ba cái lợi:

- Khớp với `range(1, miss)` của Python: gồm `1, 2, …, miss − 1`.
- Số lượng phần tử trong `[1, miss)` là `miss − 1`. Khi chưa tạo được gì thì `miss = 1` và `[1, 1)` **rỗng** — không cần trường hợp đặc biệt.
- Điều kiện dừng "đã phủ `[1, n]`" viết rất gọn: `miss > n` (vì `n` nằm trong `[1, miss)` ⇔ `n < miss`).

### 2.6 Lũy thừa của 2 và hệ nhị phân: vì sao `1, 2, 4, 8` "phủ thần kỳ"

Một hiện tượng sẽ xuất hiện liên tục trong bài: các số vá thường là `1, 2, 4, 8, 16, …` (gấp đôi). Lý do nằm ở **hệ nhị phân** (binary): mỗi số nguyên dương viết được **duy nhất** thành tổng các lũy thừa **khác nhau** của 2. Ví dụ `13 = 8 + 4 + 1` vì `bin(13) = '0b1101'` (đọc từ phải sang: bit của `1` bật, `2` tắt, `4` bật, `8` bật).

Hệ quả: với `k` số `1, 2, 4, …, 2^(k−1)` ta tạo được **mọi** số từ `1` đến `2^k − 1`. Thêm số tiếp theo `2^k` thì phạm vi **gấp đôi** thành `2^(k+1) − 1`. Mỗi lần "gấp đôi" là lý do lời giải chính chỉ cần khoảng `log2(n)` lần vá.

In [6]:
for miss in [1, 2, 5]:
    print(f"miss = {miss}: [1, miss) = {list(range(1, miss))}, có {miss - 1} số")

n = 20
for miss in [20, 21, 23]:
    print(f"miss = {miss:2}: đã phủ [1, {n}]? {miss > n}")

print("13 =", bin(13), "->", [p for p in (8, 4, 2, 1) if 13 & p])   # & : kiểm tra bit p có bật không

powers = [1, 2, 4, 8]
print("tổng tập con của", powers, "->", sorted(subset_sums(powers)))
print("đúng bằng 0..15?", subset_sums(powers) == set(range(16)))

powers.append(16)
print("thêm 16 -> phủ tới", max(subset_sums(powers)), "| liền mạch?", subset_sums(powers) == set(range(32)))

miss = 1: [1, miss) = [], có 0 số
miss = 2: [1, miss) = [1], có 1 số
miss = 5: [1, miss) = [1, 2, 3, 4], có 4 số
miss = 20: đã phủ [1, 20]? False
miss = 21: đã phủ [1, 20]? True
miss = 23: đã phủ [1, 20]? True
13 = 0b1101 -> [8, 4, 1]
tổng tập con của [1, 2, 4, 8] -> [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
đúng bằng 0..15? True
thêm 16 -> phủ tới 31 | liền mạch? True


## 3. Cách 1 — Brute force: thử mọi bộ vá, số lần vá tăng dần

### 3.1 Ý tưởng

Cách "thật thà" nhất mà người mới có thể nghĩ ra:

1. Thử `k = 0`: không vá gì, mảng có phủ `[1, n]` không?
2. Không được thì thử `k = 1`: thử **từng** số vá `p`, có số nào làm mảng phủ đủ không?
3. Không được thì thử `k = 2`: thử **mọi bộ** hai số vá…
4. Cứ thế. `k` đầu tiên có bộ vá thành công chính là **đáp án nhỏ nhất** (vì ta thử `k` từ nhỏ đến lớn).

Câu hỏi: số vá lấy giá trị trong khoảng nào? Vô hạn số nguyên dương thì thử sao hết? Nhận xét giúp giới hạn:

> **Một số vá `p > n` không bao giờ có ích**: mọi tổng chứa `p` đều `> n`, nằm ngoài phạm vi quan tâm. Vậy chỉ cần thử giá trị vá trong `[1, n]`.

Thêm nữa, thuật toán chắc chắn **dừng**: vá `1, 2, 4, …` đủ `⌈log2(n + 1)⌉` số là phủ được `[1, n]` (mục 2.6), nên `k` không bao giờ vượt quá con số đó.

### 3.2 Code

`brute_search` trả về cả `k` và **một bộ vá** cụ thể (để nhìn cho dễ); `min_patches_brute` chỉ trả về `k` giống chữ ký của đề. Biến `stats` đếm số bộ đã thử, để cảm nhận độ chậm.

In [7]:
def brute_search(nums, n, stats=None):
    k = 0
    while True:
        for patch in combinations_with_replacement(range(1, n + 1), k):
            if stats is not None:
                stats["tried"] += 1
            if covers(list(nums) + list(patch), n):
                return k, list(patch)
        k += 1


def min_patches_brute(nums, n):
    return brute_search(nums, n)[0]


for nums, n, expected in EXAMPLES:
    st = {"tried": 0}
    k, patch = brute_search(nums, n, st)
    print(f"nums = {nums!s:12} n = {n:3} -> {k} lần vá, ví dụ vá {patch}, đã thử {st['tried']} bộ  (mong đợi {expected})")
    assert k == expected

nums = [1, 3]       n =   6 -> 1 lần vá, ví dụ vá [2], đã thử 3 bộ  (mong đợi 1)
nums = [1, 5, 10]   n =  20 -> 2 lần vá, ví dụ vá [1, 3], đã thử 24 bộ  (mong đợi 2)
nums = [1, 2, 2]    n =   5 -> 0 lần vá, ví dụ vá [], đã thử 1 bộ  (mong đợi 0)


Ở Example 2 brute force tìm ra bộ vá `[2, 4]` — trùng đúng giải thích của đề (vì nó thử theo thứ tự từ điển, `[2, 4]` là bộ đầu tiên thành công).

### 3.3 Cách 1 vướng ở đâu?

Đếm chi phí:

- Mỗi lần gọi `covers` tốn khoảng `O((m + k) · n)` để xây tập tổng (mỗi phần tử duyệt tập kích thước tới `n + 1`).
- Số bộ vá phải thử ở mức `k` là `C(n + k − 1, k)`, tức khoảng `n^k / k!`.

Với `n = 2^31 − 1` và đáp án cỡ `k = 28` (sẽ gặp ở mục 6.3), con số này lớn hơn số nguyên tử trong vũ trụ. Thậm chí chỉ **một** lần gọi `covers` đã cần một tập 2 tỉ phần tử. Cell dưới đo số bộ thử khi `n` tăng dần với `nums = [1]` để thấy độ "nổ".

Dù vậy, Cách 1 **cực kỳ quý**: nó đúng theo định nghĩa, không dựa vào nhận xét thông minh nào, nên ta dùng nó làm **trọng tài** cho mọi cách sau trên các input nhỏ.

In [8]:
for n in [3, 7, 10, 15, 20, 31]:
    st = {"tried": 0}
    t0 = time.perf_counter()
    k, patch = brute_search([1], n, st)
    ms = (time.perf_counter() - t0) * 1000
    print(f"nums=[1], n = {n:2}: đáp án {k}, vá {patch!s:12}, thử {st['tried']:>6} bộ, {ms:8.1f} ms")

nums=[1], n =  3: đáp án 1, vá [2]         , thử      3 bộ,      0.0 ms
nums=[1], n =  7: đáp án 2, vá [2, 4]      , thử     18 bộ,      0.0 ms
nums=[1], n = 10: đáp án 3, vá [1, 3, 5]   , thử     88 bộ,      0.1 ms
nums=[1], n = 15: đáp án 3, vá [2, 4, 8]   , thử    288 bộ,      0.6 ms
nums=[1], n = 20: đáp án 4, vá [1, 3, 5, 10], thử   2212 bộ,      5.1 ms
nums=[1], n = 31: đáp án 4, vá [2, 4, 8, 16], thử  12455 bộ,     35.7 ms


## 4. Cách 2 — Trực giác: "thiếu số nhỏ nhất nào thì vá đúng số đó"

### 4.1 Ý tưởng

Thay vì thử mù mọi bộ vá, ta hỏi: **nếu số `v` chưa tạo được, thì phải làm gì để tạo được nó?**

Gọi `v` là số **nhỏ nhất** chưa tạo được. Để tạo `v`, ta cần một tập con có tổng đúng `v`, nên mọi phần tử của tập con đó phải `≤ v`. Mảng hiện tại không làm được, vậy **bắt buộc** phải có ít nhất một số vá `≤ v`. Trong các số `≤ v`, số "mạnh" nhất là chính `v`: vá `v` thì tạo được `v` ngay, đồng thời tạo được cả `v + s` với mọi tổng cũ `s`.

Thuật toán trực giác:

```text
lặp:
    v = số nhỏ nhất trong [1, n] chưa tạo được
    nếu không có v: dừng, trả về số lần đã vá
    vá v (thêm v vào mảng)
```

Ở bước này ta **chưa chứng minh** chiến lược "vá số nhỏ nhất còn thiếu" là tối ưu — ta sẽ kiểm tra nó với trọng tài Cách 1, rồi chứng minh ở mục 8.

In [9]:
def patches_smallest_missing(nums, n):
    # Trả về danh sách số đã vá theo chiến lược "vá số nhỏ nhất còn thiếu".
    arr = list(nums)
    patches = []
    while True:
        v = smallest_missing(arr, n)
        if v is None:
            return patches
        arr.append(v)
        patches.append(v)


def min_patches_smallest_missing(nums, n):
    return len(patches_smallest_missing(nums, n))


for nums, n, expected in EXAMPLES:
    p = patches_smallest_missing(nums, n)
    print(f"nums = {nums!s:12} n = {n:3} -> vá {p} ({len(p)} lần, mong đợi {expected})")
    assert len(p) == expected

nums = [1, 3]       n =   6 -> vá [2] (1 lần, mong đợi 1)
nums = [1, 5, 10]   n =  20 -> vá [2, 4] (2 lần, mong đợi 2)
nums = [1, 2, 2]    n =   5 -> vá [] (0 lần, mong đợi 0)


### 4.2 Nhìn kỹ tập tổng sau mỗi lần vá

Hàm `trace_smallest_missing` in tập tổng (chỉ các số `1..n`) sau mỗi lần vá, tách làm hai phần:

- **phần liền**: đoạn `[1, v)` nằm ngay trước số thiếu nhỏ nhất `v`;
- **phần rời rạc phía trên `v`**: những tổng lớn hơn `v` "lỡ" tạo được.

Hãy để ý: phần nào thực sự **quyết định** lần vá tiếp theo?

In [10]:
def describe(nums, n):
    sums = subset_sums(nums, limit=n) - {0}
    v = smallest_missing(nums, n)
    if v is None:
        return f"phủ đủ [1, {n}]"
    above = sorted(s for s in sums if s > v)
    return f"liền [1, {v}) | thiếu {v} | rời rạc phía trên: {above}"


def trace_smallest_missing(nums, n):
    arr = list(nums)
    print(f"ban đầu {arr}: {describe(arr, n)}")
    while (v := smallest_missing(arr, n)) is not None:
        arr.append(v)
        print(f"vá {v:2} -> {describe(arr, n)}")
    print("tổng số lần vá:", len(arr) - len(nums))


trace_smallest_missing([1, 5, 10], 20)
print()
trace_smallest_missing([1, 2, 31, 33], 40)

ban đầu [1, 5, 10]: liền [1, 2) | thiếu 2 | rời rạc phía trên: [5, 6, 10, 11, 15, 16]
vá  2 -> liền [1, 4) | thiếu 4 | rời rạc phía trên: [5, 6, 7, 8, 10, 11, 12, 13, 15, 16, 17, 18]
vá  4 -> phủ đủ [1, 20]
tổng số lần vá: 2

ban đầu [1, 2, 31, 33]: liền [1, 4) | thiếu 4 | rời rạc phía trên: [31, 32, 33, 34, 35, 36]
vá  4 -> liền [1, 8) | thiếu 8 | rời rạc phía trên: [31, 32, 33, 34, 35, 36, 37, 38, 39, 40]
vá  8 -> liền [1, 16) | thiếu 16 | rời rạc phía trên: [31, 32, 33, 34, 35, 36, 37, 38, 39, 40]
vá 16 -> phủ đủ [1, 40]
tổng số lần vá: 3


Ghi chú cú pháp: `while (v := smallest_missing(arr, n)) is not None` dùng **toán tử walrus** `:=` (Python 3.8+): vừa gán `v` vừa so sánh trong cùng một điều kiện.

### 4.3 Nhận xét từ trace

Với `[1, 5, 10], n = 20`:

| Thời điểm | Đoạn liền | Thiếu nhỏ nhất | Hành động |
|---|---|---:|---|
| ban đầu | `[1, 2)` | 2 | vá 2 |
| sau vá 2 | `[1, 4)` | 4 | vá 4 |
| sau vá 4 | `[1, 20]` | — | dừng, 2 lần vá |

Hai quan sát quan trọng:

1. **Phần rời rạc phía trên không bao giờ được dùng để ra quyết định.** Ta chỉ cần biết **một con số**: số nhỏ nhất còn thiếu. Vậy mà Cách 2 đang tốn công xây cả tập tổng kích thước tới `n`.
2. Ở ví dụ thứ hai, các số vá là `4, 8, 16, …` — cứ mỗi lần vá, đoạn liền **gấp đôi** (`[1,4) → [1,8) → [1,16)…`), và khi đoạn liền đủ dài thì các số có sẵn `31, 33` được "nuốt" vào miễn phí.

### 4.4 Cách 2 vướng ở đâu?

Mỗi lần gọi `smallest_missing` xây lại tập tổng: `O((m + p) · n)` thời gian và `O(n)` bộ nhớ (`p` = số lần vá). Với `n = 2^31 − 1`, chỉ riêng một tập 2 tỉ số nguyên đã cần hàng chục GB RAM. Cách 2 đúng hướng về **ý tưởng** nhưng sai về **cách lưu trữ**. Ở mục 7.1 ta sẽ đối chiếu nó với trọng tài Cách 1 trên dữ liệu ngẫu nhiên.

In [11]:
def random_case(rng, max_len=4, max_val=10, max_n=12):
    m = rng.randint(1, max_len)
    nums = sorted(rng.randint(1, max_val) for _ in range(m))
    n = rng.randint(1, max_n)
    return nums, n


rng = random.Random(330)
for _ in range(150):
    nums, n = random_case(rng)
    assert min_patches_smallest_missing(nums, n) == min_patches_brute(nums, n), (nums, n)
print("Cách 2 khớp trọng tài Cách 1 trên 150 test ngẫu nhiên nhỏ")

Cách 2 khớp trọng tài Cách 1 trên 150 test ngẫu nhiên nhỏ


## 5. Nhận xét then chốt: thay cả tập tổng bằng một biến `miss`

Mục tiêu: tìm cách cập nhật "số nhỏ nhất còn thiếu" **mà không cần** tập tổng. Ta sẽ xét các phần tử theo **thứ tự tăng dần** (đề đã cho `nums` sắp xếp sẵn — đây là lý do đề cho như vậy).

### 5.1 Quy tắc mở rộng: thêm `x ≤ miss` thì `[1, miss)` thành `[1, miss + x)`

> **Nhận xét 1.** Giả sử các phần tử đang có tạo được **mọi** số trong `[1, miss)`. Thêm một phần tử `x` với `1 ≤ x ≤ miss`. Khi đó tạo được **mọi** số trong `[1, miss + x)`.

Vì sao? Lấy một số `v` bất kỳ trong `[1, miss + x)`:

- Nếu `v < miss`: tạo được từ trước, không cần `x`.
- Nếu `miss ≤ v < miss + x`: viết `v = x + (v − x)`. Ta có `v − x ≥ miss − x ≥ 0` (vì `x ≤ miss`) và `v − x < miss`. Vậy `v − x` nằm trong `[0, miss)` nên tạo được bằng các phần tử cũ (số `0` = tập rỗng), cộng thêm `x` là ra `v`. Phần tử `x` mới chưa nằm trong tập cũ nên không bị dùng hai lần.

**Ví dụ số:** `[1, 1, 2]` tạo được `1, 2, 3, 4`, tức `miss = 5`. Thêm `x = 3 ≤ 5`:

| v | cách tạo |
|---:|---|
| 5 | `3 + 2` (2 tạo từ cũ) |
| 6 | `3 + 3` → `3 + (1 + 2)` |
| 7 | `3 + 4` → `3 + (1 + 1 + 2)` |

Đoạn mới là `[1, 8)`, đúng bằng `[1, 5 + 3)`. Số `8` thì không tạo được vì tổng tất cả `1 + 1 + 2 + 3 = 7`.

In [12]:
def contiguous_miss(nums):
    # Số nhỏ nhất (>= 1) không tạo được, tính bằng tập tổng (để kiểm chứng).
    sums = subset_sums(nums)
    v = 1
    while v in sums:
        v += 1
    return v


base = [1, 1, 2]
miss = contiguous_miss(base)
print("base =", base, "-> miss =", miss)
for x in [1, 3, 5, 6, 9]:
    new_miss = contiguous_miss(base + [x])
    rule = miss + x if x <= miss else miss
    print(f"thêm x = {x}: miss thật = {new_miss:2} | quy tắc dự đoán = {rule:2} | x <= miss? {x <= miss}")

base = [1, 1, 2] -> miss = 5
thêm x = 1: miss thật =  6 | quy tắc dự đoán =  6 | x <= miss? True
thêm x = 3: miss thật =  8 | quy tắc dự đoán =  8 | x <= miss? True
thêm x = 5: miss thật = 10 | quy tắc dự đoán = 10 | x <= miss? True
thêm x = 6: miss thật =  5 | quy tắc dự đoán =  5 | x <= miss? False
thêm x = 9: miss thật =  5 | quy tắc dự đoán =  5 | x <= miss? False


Bảng trên cho thấy:

- Với `x = 1, 3, 5` (đều `≤ miss = 5`): `miss` mới đúng bằng `miss + x`.
- Với `x = 6, 9` (`> miss`): `miss` **giữ nguyên** `5` — số `5` vẫn không tạo được dù thêm số lớn.

Trường hợp thứ hai không phải ngẫu nhiên. Ta phát biểu thành nhận xét.

### 5.2 Nếu phần tử tiếp theo `> miss` thì `miss` không bao giờ tạo được (trừ khi vá)

> **Nhận xét 2.** Khi duyệt các phần tử theo thứ tự **tăng dần** và chỉ "nuốt" những phần tử `≤ miss`, ta luôn có:
>
> **`miss = 1 + (tổng các phần tử đã nuốt)`**.
>
> Vì vậy, nếu phần tử tiếp theo `nums[i] > miss` thì **không có cách nào** tạo ra `miss` bằng các phần tử của `nums`.

Đẳng thức `miss = 1 + tổng` đúng vì: lúc đầu chưa nuốt gì, `miss = 1 = 1 + 0`; mỗi lần nuốt `x` thì `miss` tăng đúng `x` và tổng cũng tăng đúng `x`.

Giờ thử tạo `miss`:

- Dùng **chỉ** các phần tử đã nuốt: tổng lớn nhất có thể là `miss − 1 < miss`. Không đủ.
- Dùng bất kỳ phần tử nào chưa nuốt: chúng đều `≥ nums[i] > miss` (vì mảng tăng dần), một mình nó đã vượt `miss`. Quá nhiều.

Vậy `miss` **bắt buộc** phải đến từ một số vá. Cell dưới in cột "tổng đã nuốt" để thấy đẳng thức trên với `[1, 2, 2, 7, 20]`.

### 5.3 Nên vá số nào? Vá chính `miss` để đoạn phủ gấp đôi

Khi bị kẹt (phần tử tiếp theo `> miss` hoặc đã hết phần tử) mà `miss ≤ n`, ta buộc phải vá một số `y`. Theo Nhận xét 1:

- Vá `y ≤ miss` → đoạn phủ thành `[1, miss + y)`. `y` càng lớn thì đoạn càng dài, **lớn nhất là `y = miss`** → `[1, 2·miss)`, tức **gấp đôi**.
- Vá `y > miss` → `miss` vẫn chưa tạo được (lý do y hệt Nhận xét 2), coi như **phí** lần vá này cho mục tiêu trước mắt.

Cell dưới minh họa với `miss = 8` (ví dụ đang có `[1, 2, 4]`), thử mọi `y` từ 1 đến 10.

In [13]:
have = [1, 2, 4]          # phủ [1, 8)
miss = contiguous_miss(have)
print("đang có", have, "-> miss =", miss)
for y in range(1, 11):
    new_miss = contiguous_miss(have + [y])
    note = "  <- tốt nhất: gấp đôi" if y == miss else ("  (phí: 8 vẫn thiếu)" if y > miss else "")
    print(f"vá y = {y:2}: phủ [1, {new_miss:2}){note}")

đang có [1, 2, 4] -> miss = 8
vá y =  1: phủ [1,  9)
vá y =  2: phủ [1, 10)
vá y =  3: phủ [1, 11)
vá y =  4: phủ [1, 12)
vá y =  5: phủ [1, 13)
vá y =  6: phủ [1, 14)
vá y =  7: phủ [1, 15)
vá y =  8: phủ [1, 16)  <- tốt nhất: gấp đôi
vá y =  9: phủ [1,  8)  (phí: 8 vẫn thiếu)
vá y = 10: phủ [1,  8)  (phí: 8 vẫn thiếu)


### 5.4 Ghép lại: thuật toán tham lam (greedy)

**Greedy** (tham lam) là kiểu thuật toán ở mỗi bước chọn phương án **tốt nhất trước mắt** mà không quay lại xét lại. Greedy không phải lúc nào cũng đúng — vì vậy luôn phải **chứng minh** (mục 8) hoặc ít nhất kiểm tra với brute force.

Từ hai nhận xét, ta chỉ cần giữ ba biến: `miss` (đoạn `[1, miss)` đã phủ), `i` (phần tử tiếp theo của `nums`), `patches` (số lần vá). Mỗi vòng lặp, khi `miss ≤ n` (còn thiếu):

1. Nếu còn phần tử và `nums[i] ≤ miss`: **nuốt miễn phí** — `miss += nums[i]`, `i += 1`.
2. Ngược lại: **vá chính `miss`** — `miss += miss`, `patches += 1`.

Thứ tự ưu tiên quan trọng: luôn dùng hàng có sẵn trước khi tốn một lần vá. Lần vá chỉ xảy ra khi thật sự bị kẹt.

So với Cách 2: cả tập tổng kích thước `O(n)` được thay bằng **một số nguyên** `miss`, và mỗi lần vá `miss` **ít nhất gấp đôi**, nên số vòng lặp chỉ khoảng `m + log2(n)`.

## 6. Cách 3 — Greedy với `miss` (lời giải chính)

### 6.1 Code từng đoạn

- `miss = 1`: chưa tạo được gì, đoạn `[1, 1)` rỗng, số nhỏ nhất còn thiếu là `1`.
- `while miss <= n`: còn ít nhất số `miss` (`≤ n`) chưa phủ. Khi `miss > n` thì `[1, n] ⊆ [1, miss)`, xong.
- Nhánh `if`: còn phần tử (`i < len(nums)`) **và** phần tử đó `≤ miss` (dấu `<=`: phần tử **bằng** `miss` vẫn dùng được, vì `miss = miss + 0`).
- Nhánh `else`: vá `miss`, đoạn phủ gấp đôi.

Lưu ý: các phần tử `nums[i] > n` sẽ không bao giờ được nuốt (vòng lặp dừng trước), hoàn toàn đúng vì chúng không giúp tạo số nào `≤ n`.

In [14]:
def min_patches_greedy(nums, n):
    miss = 1          # mọi số trong [1, miss) đã tạo được
    i = 0             # phần tử tiếp theo của nums chưa xét
    patches = 0
    while miss <= n:
        if i < len(nums) and nums[i] <= miss:
            miss += nums[i]       # nuốt miễn phí: [1, miss) -> [1, miss + nums[i])
            i += 1
        else:
            miss += miss          # vá chính miss: [1, miss) -> [1, 2 * miss)
            patches += 1
    return patches


for nums, n, expected in EXAMPLES:
    got = min_patches_greedy(nums, n)
    print(f"nums = {nums!s:12} n = {n:3} -> {got} (mong đợi {expected})")
    assert got == expected

nums = [1, 3]       n =   6 -> 1 (mong đợi 1)
nums = [1, 5, 10]   n =  20 -> 2 (mong đợi 2)
nums = [1, 2, 2]    n =   5 -> 0 (mong đợi 0)


### 6.2 Trace bằng tay: `nums = [1, 5, 10]`, `n = 20`

Mỗi dòng là một vòng lặp. Cột "điều kiện" là phép so `nums[i] <= miss` (hoặc "hết" nếu `i = len(nums)`).

| Bước | `miss` trước | `i` | `nums[i]` | Điều kiện | Hành động | `miss` sau | Đoạn đã phủ | `patches` |
|---:|---:|---:|---:|---|---|---:|---|---:|
| 1 | 1 | 0 | 1 | `1 <= 1` ✔ | nuốt 1 | 2 | `[1, 2)` | 0 |
| 2 | 2 | 1 | 5 | `5 <= 2` ✘ | **vá 2** | 4 | `[1, 4)` | 1 |
| 3 | 4 | 1 | 5 | `5 <= 4` ✘ | **vá 4** | 8 | `[1, 8)` | 2 |
| 4 | 8 | 1 | 5 | `5 <= 8` ✔ | nuốt 5 | 13 | `[1, 13)` | 2 |
| 5 | 13 | 2 | 10 | `10 <= 13` ✔ | nuốt 10 | 23 | `[1, 23)` | 2 |
| — | 23 | 3 | — | `23 <= 20` ✘ | dừng vòng lặp | | `[1, 23) ⊇ [1, 20]` | **2** |

Điểm đáng chú ý:

- Ở bước 2 và 3, số `5` **chưa dùng được** vì đoạn phủ còn quá ngắn (`[1, 2)`, rồi `[1, 4)`): không có cách tạo số `2`, rồi số `4`. Hai số vá `2, 4` đúng bằng giải thích của đề.
- Ở bước 4, đoạn `[1, 8)` đủ dài để "đỡ" số `5`: mọi `v` trong `[8, 13)` viết được thành `5 + (v − 5)` với `v − 5` trong `[3, 8)`.
- Cuối cùng `miss = 23 = 1 + (1 + 2 + 4 + 5 + 10)` — đúng đẳng thức "1 + tổng đã nuốt" của Nhận xét 2 (tính cả số vá).

Hàm `trace_greedy` in đúng bảng trên.

In [15]:
def trace_greedy(nums, n):
    miss, i, patches, step = 1, 0, 0, 0
    print(f"{'bước':>4} | {'miss':>10} | {'nums[i]':>7} | {'hành động':<14} | {'miss sau':>10} | patches")
    while miss <= n:
        step += 1
        cur = nums[i] if i < len(nums) else None
        before = miss
        if cur is not None and cur <= miss:
            miss += cur
            i += 1
            action = f"nuốt {cur}"
        else:
            miss += miss
            patches += 1
            action = f"VÁ {before}"
        shown = "hết" if cur is None else cur
        print(f"{step:>4} | {before:>10} | {shown!s:>7} | {action:<14} | {miss:>10} | {patches}")
    print(f"dừng: miss = {miss} > n = {n}  ->  đáp án {patches}")
    return patches


trace_greedy([1, 5, 10], 20)

bước |       miss | nums[i] | hành động      |   miss sau | patches
   1 |          1 |       1 | nuốt 1         |          2 | 0
   2 |          2 |       5 | VÁ 2           |          4 | 1
   3 |          4 |       5 | VÁ 4           |          8 | 2
   4 |          8 |       5 | nuốt 5         |         13 | 2
   5 |         13 |      10 | nuốt 10        |         23 | 2
dừng: miss = 23 > n = 20  ->  đáp án 2


2

### 6.3 Trace một ví dụ có `n` khổng lồ

`nums = [1, 2, 31, 33]`, `n = 2147483647` (`= 2^31 − 1`, giá trị lớn nhất của constraint). Đây là một test nổi tiếng của bài. Hãy để ý ba pha:

1. Nuốt `1, 2` → `miss = 4`.
2. Vá `4, 8, 16` → `miss = 32`, đủ để nuốt `31` (→ `63`) rồi `33` (→ `96`).
3. Hết phần tử: chỉ còn **gấp đôi liên tục** `96 → 192 → … ` cho tới khi vượt `n`.

Cách 1 và Cách 2 không thể chạy nổi input này, còn Cách 3 chỉ cần 32 vòng lặp.

Kiểm tra pha 3 bằng toán: sau khi nuốt `33`, `miss = 96`. Cần số lần gấp đôi `t` nhỏ nhất để `96 · 2^t > 2147483647`:

- `96 · 2^24 = 1 610 612 736 ≤ n` → chưa đủ;
- `96 · 2^25 = 3 221 225 472 > n` → đủ.

Vậy pha 3 có `25` lần vá, cộng `3` lần ở pha 2: tổng **28**. Để ý `miss` cuối cùng vượt `2^31 − 1`: Python không bị tràn số (int của Python lớn tùy ý), nhưng nếu viết bằng Java/C++ thì phải dùng kiểu `long` (xem mục Lỗi thường gặp).

In [16]:
ans_big = trace_greedy([1, 2, 31, 33], 2147483647)

bước |       miss | nums[i] | hành động      |   miss sau | patches
   1 |          1 |       1 | nuốt 1         |          2 | 0
   2 |          2 |       2 | nuốt 2         |          4 | 0
   3 |          4 |      31 | VÁ 4           |          8 | 1
   4 |          8 |      31 | VÁ 8           |         16 | 2
   5 |         16 |      31 | VÁ 16          |         32 | 3
   6 |         32 |      31 | nuốt 31        |         63 | 3
   7 |         63 |      33 | nuốt 33        |         96 | 3
   8 |         96 |     hết | VÁ 96          |        192 | 4
   9 |        192 |     hết | VÁ 192         |        384 | 5
  10 |        384 |     hết | VÁ 384         |        768 | 6
  11 |        768 |     hết | VÁ 768         |       1536 | 7
  12 |       1536 |     hết | VÁ 1536        |       3072 | 8
  13 |       3072 |     hết | VÁ 3072        |       6144 | 9
  14 |       6144 |     hết | VÁ 6144        |      12288 | 10
  15 |      12288 |     hết | VÁ 12288       |      24576 | 11


## 7. Nộp LeetCode: `class Solution`

Chuyển `min_patches_greedy` vào đúng signature của đề. Giữ nguyên logic, chỉ đổi tên biến cho gọn. Sau đó test bằng `assert`:

- 3 ví dụ của đề;
- các edge case: `n = 1`; `nums` đã thừa đủ (`[1, 2, 4, 8]` với `n = 15` → 0); phần tử đầu tiên lớn (`[10]`, `n = 12`); toàn số lớn hơn `n` (`[100]`, `n = 5`); nhiều số `1` trùng; `n` lớn nhất với `nums = [1]`.

Cuối cell còn một bài **thử sức với input lớn nhất**: `m = 1000` phần tử ngẫu nhiên trong `[1, 10^4]`, `n = 2^31 − 1`. Số vòng lặp tối đa khoảng `1000 + 31`, nên phải chạy tức thì (mảng `big` này dùng lại ở mục 8.3).

In [17]:
class Solution:
    def minPatches(self, nums: List[int], n: int) -> int:
        miss, i, patches = 1, 0, 0
        while miss <= n:
            if i < len(nums) and nums[i] <= miss:
                miss += nums[i]
                i += 1
            else:
                miss += miss
                patches += 1
        return patches


sol = Solution()
for nums, n, expected in EXAMPLES:
    assert sol.minPatches(nums, n) == expected

assert sol.minPatches([1], 1) == 0
assert sol.minPatches([2], 1) == 1                   # phải vá 1
assert sol.minPatches([1, 2, 4, 8], 15) == 0
assert sol.minPatches([1, 2, 4, 8], 16) == 1         # vá 16
assert sol.minPatches([10], 12) == 4                 # vá 1, 2, 4, 8
assert sol.minPatches([100], 5) == 3                 # vá 1, 2, 4; số 100 vô dụng
assert sol.minPatches([1] * 1000, 1000) == 0
assert sol.minPatches([1, 2, 31, 33], 2147483647) == 28
assert sol.minPatches([1], 2147483647) == 30         # 1 có sẵn, vá 2, 4, ..., 2^30
print("Tất cả test của class Solution đều đúng")

random.seed(1)
big = sorted(random.randint(1, 10**4) for _ in range(1000))
t0 = time.perf_counter()
ans = sol.minPatches(big, 2**31 - 1)
elapsed_ms = (time.perf_counter() - t0) * 1000
print("đáp án:", ans, "| thời gian: %.3f ms" % elapsed_ms)
print("chạy dưới 10 ms?", elapsed_ms < 10)

Tất cả test của class Solution đều đúng
đáp án: 13 | thời gian: 0.130 ms
chạy dưới 10 ms? True


### 7.1 Đối chiếu chéo ba cách trên dữ liệu ngẫu nhiên

Sinh 300 test nhỏ với `random.seed` cố định (để lần chạy nào cũng ra cùng dữ liệu): `nums` dài 1–5, giá trị 1–12, `n` từ 1 đến 14. So sánh **Cách 1 (trọng tài)**, **Cách 2** và **Cách 3**. Đồng thời kiểm tra một điều thú vị: danh sách số vá của Cách 2 có đúng là `miss` tại các bước vá của Cách 3 không.

In [18]:
def greedy_patch_list(nums, n):
    miss, i, patches = 1, 0, []
    while miss <= n:
        if i < len(nums) and nums[i] <= miss:
            miss += nums[i]
            i += 1
        else:
            patches.append(miss)
            miss += miss
    return patches


random.seed(330)
worst = (0, None)
for _ in range(300):
    m = random.randint(1, 5)
    nums = sorted(random.randint(1, 12) for _ in range(m))
    n = random.randint(1, 14)
    a1 = min_patches_brute(nums, n)
    a2 = patches_smallest_missing(nums, n)
    a3 = sol.minPatches(nums, n)
    assert a1 == len(a2) == a3, (nums, n, a1, a2, a3)
    assert a2 == greedy_patch_list(nums, n)          # Cách 2 vá đúng các giá trị miss
    worst = max(worst, (a1, (nums, n)))
print("300 test: Cách 1 == Cách 2 == Cách 3, và Cách 2 vá đúng dãy miss của Cách 3")
print("test cần vá nhiều nhất:", worst)

300 test: Cách 1 == Cách 2 == Cách 3, và Cách 2 vá đúng dãy miss của Cách 3
test cần vá nhiều nhất: (4, ([12], 14))


## 8. Vì sao Cách 3 đúng và tối ưu?

### 8.1 Invariant (bất biến) của vòng lặp

Trước mỗi vòng lặp, hai điều sau luôn đúng:

- **(I1)** Các phần tử đã xét (gồm `nums[0..i−1]` và các số đã vá) tạo được **mọi** số trong `[1, miss)`.
- **(I2)** `miss = 1 + (tổng các phần tử đã xét)`.

Lúc đầu: chưa xét gì, `miss = 1`, `[1, 1)` rỗng, `1 = 1 + 0`. ✔

Mỗi bước giữ nguyên invariant:

- Nhánh nuốt `x = nums[i] ≤ miss`: theo Nhận xét 1, đoạn phủ thành `[1, miss + x)`; tổng tăng `x`, `miss` tăng `x`. ✔
- Nhánh vá `y = miss`: cũng là thêm một phần tử `y ≤ miss`, nên y hệt. ✔

**Dừng và đúng:** mỗi vòng `miss` tăng ít nhất 1, nên vòng lặp kết thúc; khi kết thúc `miss > n` nên theo (I1) mọi số `1..n` tạo được. Vậy số lần vá greedy trả về là **khả thi**.

### 8.2 Tối ưu: không cách vá nào dùng ít lần hơn

Ba ý, ghép lại thành lập luận "**greedy luôn đi trước**" (greedy stays ahead):

**(a) Mọi mảng cuối cùng đều được "đọc" bằng phép quét tăng dần.** Lấy một lời giải bất kỳ (nums + các số vá), sắp xếp tăng dần rồi quét như Nhận xét 2. Nếu quét bị kẹt ở `miss ≤ n` thì (theo Nhận xét 2) số `miss` không tạo được → lời giải không hợp lệ. Vậy lời giải hợp lệ ⇔ phép quét của nó đi tới `miss > n`. Mọi lời giải đều là một **dãy bước** "nuốt `nums[i]`" hoặc "nuốt một số vá `y ≤ miss`".

**(b) `miss` lớn hơn thì không bao giờ thiệt (tính đơn điệu).** So hai trạng thái có cùng con trỏ `i`, `miss_A ≤ miss_B`. Bất kỳ bước nào hợp lệ ở A cũng hợp lệ ở B: nuốt `nums[i] ≤ miss_A ≤ miss_B` được, vá `y ≤ miss_A ≤ miss_B` được, và sau bước đó vẫn `miss_A ≤ miss_B` (cả hai cộng cùng một số). Nên B luôn tới đích **không chậm hơn** A, với không nhiều lần vá hơn.

**(c) Mỗi quyết định của greedy đều không tệ hơn mọi lựa chọn khác.**

- Khi `nums[i] ≤ miss`, greedy nuốt miễn phí. Mọi lời giải cuối cùng đều phải nuốt `nums[i]` ở một lúc nào đó hoặc bỏ nó; nuốt ngay chỉ làm `miss` lớn hơn mà không tốn lần vá → theo (b) không thiệt.
- Khi bị kẹt, theo Nhận xét 2 **mọi** lời giải đều phải vá một số `y ≤ miss` tại đây (không thì `miss` không bao giờ tạo được). Greedy chọn `y = miss`, cho `miss` mới **lớn nhất** trong mọi lựa chọn → theo (b) không thiệt.

Vậy sau mỗi lần vá thứ `p`, `miss` của greedy luôn `≥` `miss` của bất kỳ lời giải nào sau `p` lần vá. Lời giải tối ưu dùng `p*` lần vá để vượt `n` thì greedy cũng vượt `n` trong tối đa `p*` lần vá. ∎

Cell dưới kiểm chứng ý (a) bằng thực nghiệm: phép quét tăng dần cho ra đúng "số nhỏ nhất không tạo được" tính bằng tập tổng, trên 500 multiset ngẫu nhiên.

In [19]:
def scan_miss(values):
    miss = 1
    for x in sorted(values):
        if x > miss:
            break
        miss += x
    return miss


random.seed(8)
for _ in range(500):
    vals = [random.randint(1, 20) for _ in range(random.randint(0, 7))]
    assert scan_miss(vals) == contiguous_miss(vals), vals
print("Phép quét tăng dần == số nhỏ nhất không tạo được (tập tổng) trên 500 multiset ngẫu nhiên")

Phép quét tăng dần == số nhỏ nhất không tạo được (tập tổng) trên 500 multiset ngẫu nhiên


### 8.3 Độ phức tạp

Gọi `m = len(nums)`.

- **Thời gian `O(m + log n)`**: mỗi vòng lặp hoặc tăng `i` (tối đa `m` lần), hoặc **gấp đôi** `miss`. `miss` bắt đầu từ `1` và dừng khi `> n`, mà mỗi lần vá `miss` ít nhất gấp đôi, nên số lần vá `≤ ⌊log2 n⌋ + 1` (tối đa **31** lần với `n < 2^31`).
- **Bộ nhớ `O(1)`**: chỉ ba biến nguyên.

Cell dưới đếm số vòng lặp thật trên vài input và so với chặn `m + ⌊log2 n⌋ + 1`.

In [20]:
def count_loops(nums, n):
    miss, i, loops = 1, 0, 0
    while miss <= n:
        loops += 1
        if i < len(nums) and nums[i] <= miss:
            miss += nums[i]
            i += 1
        else:
            miss += miss
    return loops


for nums, n in [([1, 5, 10], 20), ([1, 2, 31, 33], 2**31 - 1), ([1], 2**31 - 1), (big, 2**31 - 1)]:
    bound = len(nums) + n.bit_length()          # n.bit_length() = floor(log2 n) + 1
    label = f"{nums}" if len(nums) <= 4 else f"<{len(nums)} số ngẫu nhiên>"
    print(f"{label:22} n = {n:>10}: {count_loops(nums, n):4} vòng  <=  chặn {bound}")

[1, 5, 10]             n =         20:    5 vòng  <=  chặn 8
[1, 2, 31, 33]         n = 2147483647:   32 vòng  <=  chặn 35
[1]                    n = 2147483647:   31 vòng  <=  chặn 32
<1000 số ngẫu nhiên>   n = 2147483647: 1013 vòng  <=  chặn 1031


## 9. Cách 4 — So sánh: vá `miss` đối đầu với các chiến lược vá khác

Chứng minh ở mục 8 nói vá `miss` là tốt nhất. Nhưng "nghe" thì dễ, "thấy" thì nhớ lâu. Ta viết một bộ **mô phỏng chung** (dùng tập tổng, chỉ cho `n` nhỏ): mỗi bước tìm danh sách các số còn thiếu trong `[1, n]`, rồi để một **chiến lược** quyết định vá số nào. Các chiến lược đem ra thi:

| Chiến lược | Vá số nào | Suy nghĩ "nghe có lý" của người mới |
|---|---|---|
| `vá miss` | số nhỏ nhất còn thiếu (= greedy) | — |
| `vá 1` | luôn vá số `1` | "số 1 an toàn, tạo được mọi thứ dần dần" |
| `vá lớn nhất thiếu` | số **lớn nhất** còn thiếu `≤ n` | "lấp từ trên xuống, số lớn đóng góp nhiều" |
| `vá nửa miss` | `max(1, miss // 2)` | "vá nhỏ hơn cho chắc ăn" |
| `vá miss − 1` | số thiếu nhỏ nhất **trừ 1** (ít nhất là 1) | "vá sát dưới lỗ hổng là đủ lấp nó rồi" |

Đồng thời so với **đáp án tối ưu** của trọng tài Cách 1.

In [21]:
def simulate(nums, n, choose):
    arr, patches = list(nums), []
    while True:
        sums = subset_sums(arr, limit=n)
        missing = [v for v in range(1, n + 1) if v not in sums]
        if not missing:
            return patches
        y = choose(missing, n)
        arr.append(y)
        patches.append(y)


STRATEGIES = {
    "vá miss": lambda missing, n: missing[0],
    "vá 1": lambda missing, n: 1,
    "vá lớn nhất thiếu": lambda missing, n: missing[-1],
    "vá nửa miss": lambda missing, n: max(1, missing[0] // 2),
    "vá miss − 1": lambda missing, n: max(1, missing[0] - 1),
}

cases = [([1], 7), ([1, 2], 15), ([1, 5, 10], 20), ([3], 14)]
print(f"{'input':18}" + "".join(f"{name:>19}" for name in STRATEGIES) + f"{'tối ưu':>8}")
for nums, n in cases:
    row = f"{str(nums) + ', n=' + str(n):18}"
    for name, choose in STRATEGIES.items():
        row += f"{len(simulate(nums, n, choose)):>19}"
    print(row + f"{min_patches_brute(nums, n):>8}")

print()
print("Chi tiết với nums = [1, 2], n = 15:")
for name, choose in STRATEGIES.items():
    p = simulate([1, 2], 15, choose)
    print(f"{name:18}: vá {p}  ({len(p)} lần)")
print("tối ưu (trọng tài):", brute_search([1, 2], 15))

input                         vá miss               vá 1  vá lớn nhất thiếu        vá nửa miss        vá miss − 1  tối ưu
[1], n=7                            2                  6                  6                  4                  3       2
[1, 2], n=15                        2                 12                 12                  4                  3       2
[1, 5, 10], n=20                    2                  4                 13                  3                  3       2
[3], n=14                           4                 11                 13                  5                  4       4

Chi tiết với nums = [1, 2], n = 15:
vá miss           : vá [4, 8]  (2 lần)
vá 1              : vá [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]  (12 lần)
vá lớn nhất thiếu : vá [15, 14, 13, 12, 11, 10, 9, 8, 7, 6, 5, 4]  (12 lần)
vá nửa miss       : vá [2, 3, 4, 6]  (4 lần)
vá miss − 1       : vá [3, 6, 12]  (3 lần)
tối ưu (trọng tài): (2, [4, 8])


Chỉ cột `vá miss` luôn bằng cột `tối ưu`. Phần thứ hai của output in cụ thể các số đã vá ở một phản ví dụ, `nums = [1, 2]`, `n = 15` (lúc đầu đang phủ `[1, 4)`).

Đọc kết quả:

- `vá miss`: `4` rồi `8` → `[1, 2, 4, 8]` phủ `[1, 16)`. Mỗi lần vá **gấp đôi** đoạn phủ.
- `vá 1`: mỗi lần chỉ nhích đoạn phủ thêm `1` → cần tới `12` lần. Thua theo cấp số: `O(n)` lần vá so với `O(log n)`.
- `vá lớn nhất thiếu`: vá `15`, `14`, … tạo được nhiều tổng "lẻ tẻ" phía trên, nhưng lỗ hổng nhỏ nhất (`4`) vẫn còn đó. Lỗ hổng nhỏ là thứ **chỉ** số nhỏ lấp được.
- `vá nửa miss`, `vá miss − 1`: vẫn lấp được lỗ hổng `miss` (vì `miss − y` đã tạo được), nhưng mỗi bước đều kém `vá miss` một chút; tích lũy lại thành thêm lần vá.

### 9.1 "Thử mọi lựa chọn" ở một bước bị kẹt

Một thí nghiệm sắc nét hơn: với `nums = [1, 2]`, `n = 15`, greedy bị kẹt ở `miss = 4`. Giả sử lần vá **đầu tiên** ta chọn số `y` tùy ý, các lần vá sau chơi **tối ưu** (dùng trọng tài Cách 1). Tổng số lần vá là bao nhiêu?

In [22]:
nums, n = [1, 2], 15
print("miss hiện tại =", scan_miss(nums))
for y in range(1, 8):
    rest_k, rest_patch = brute_search(nums + [y], n)
    total = 1 + rest_k
    mark = "  <- y = miss" if y == scan_miss(nums) else ""
    print(f"lần vá đầu y = {y}: phủ [1, {scan_miss(nums + [y]):2}), vá tiếp tối ưu {rest_patch} -> tổng {total}{mark}")

miss hiện tại = 4
lần vá đầu y = 1: phủ [1,  5), vá tiếp tối ưu [3, 8] -> tổng 3
lần vá đầu y = 2: phủ [1,  6), vá tiếp tối ưu [2, 8] -> tổng 3
lần vá đầu y = 3: phủ [1,  7), vá tiếp tối ưu [1, 8] -> tổng 3
lần vá đầu y = 4: phủ [1,  8), vá tiếp tối ưu [8] -> tổng 2  <- y = miss
lần vá đầu y = 5: phủ [1,  4), vá tiếp tối ưu [1, 6] -> tổng 3
lần vá đầu y = 6: phủ [1,  4), vá tiếp tối ưu [1, 5] -> tổng 3
lần vá đầu y = 7: phủ [1,  4), vá tiếp tối ưu [1, 4] -> tổng 3


Chỉ `y = 4 = miss` cho tổng `2`. Mọi lựa chọn khác — nhỏ hơn (`1, 2, 3`) hay lớn hơn (`5, 6, 7`) — đều tốn `3`. Đây chính là ý (c) trong chứng minh: tại bước bị kẹt, `y = miss` cho đoạn phủ dài nhất, mà đoạn phủ dài hơn thì không bao giờ thiệt.

### 9.2 Bảng so sánh các cách

Gọi `m = len(nums)`, `p` = số lần vá (`p ≤ log2(n) + 1`).

| Cách | Ý tưởng | Thời gian | Bộ nhớ | Dùng khi |
|---|---|---|---|---|
| 1. Brute force | thử mọi bộ `k` số vá trong `[1, n]`, `k` tăng dần | `≈ C(n+p−1, p) · (m+p) · n` | `O(n)` | trọng tài cho `n ≤ ~30` |
| 2. Vá số nhỏ nhất còn thiếu | xây tập tổng, tìm lỗ nhỏ nhất, vá nó | `O(p · (m + p) · n)` | `O(n)` | hiểu trực giác, `n` cỡ vài nghìn |
| 3. Greedy `miss` | nuốt `nums[i] ≤ miss`, kẹt thì vá `miss` | **`O(m + log n)`** | **`O(1)`** | **nộp LeetCode** |
| 4. Chiến lược vá khác | vá 1 / vá lớn nhất / vá nửa… | — | — | phản ví dụ, **không** tối ưu |

Bước nhảy quan trọng nhất là từ Cách 2 sang Cách 3: nhận ra rằng tập tổng luôn chứa đoạn liền `[1, miss)` và **chỉ `miss`** quyết định bước tiếp theo, nên cả tập được nén thành một số nguyên.

## 10. Lỗi thường gặp

Mỗi lỗi dưới đây: code sai → input làm lộ lỗi → vì sao sai → cách sửa. Ta so với `sol.minPatches` (đã kiểm chứng).

### Lỗi 1: dùng `nums[i] < miss` thay vì `nums[i] <= miss`

Phần tử **bằng** `miss` vẫn nuốt được: đoạn `[1, miss)` cộng thêm `miss` tạo được đúng `miss` (tập con chỉ gồm nó) và mở rộng thành `[1, 2·miss)`. Viết `<` sẽ bỏ phí phần tử đó và tốn một lần vá vô ích.

In [23]:
def wrong_strict(nums, n):
    miss, i, patches = 1, 0, 0
    while miss <= n:
        if i < len(nums) and nums[i] < miss:        # SAI: phải là <=
            miss += nums[i]
            i += 1
        else:
            miss += miss
            patches += 1
    return patches


for nums, n in [([1, 2, 4], 7), ([1, 3], 6)]:
    print(f"nums = {nums}, n = {n}: bản sai = {wrong_strict(nums, n)}, đúng = {sol.minPatches(nums, n)}")

nums = [1, 2, 4], n = 7: bản sai = 1, đúng = 0
nums = [1, 3], n = 6: bản sai = 2, đúng = 1


Với `[1, 2, 4]`, `n = 7`: lúc đầu `miss = 1`, bản sai thấy `1 < 1` là sai nên đi vá `1` — dù mảng có sẵn số `1`! Kết quả trả `1` thay vì `0`.

**Sửa:** điều kiện nuốt là `nums[i] <= miss`.

### Lỗi 2: điều kiện lặp `while miss < n` thay vì `while miss <= n`

`miss` là số **chưa** tạo được. Khi `miss == n` nghĩa là chính `n` còn thiếu, phải tiếp tục. Viết `miss < n` sẽ dừng sớm đúng một bước.

In [24]:
def wrong_stop(nums, n):
    miss, i, patches = 1, 0, 0
    while miss < n:                                  # SAI: phải là <=
        if i < len(nums) and nums[i] <= miss:
            miss += nums[i]
            i += 1
        else:
            miss += miss
            patches += 1
    return patches


for nums, n in [([2], 1), ([1, 2], 4), ([1, 5, 10], 8)]:
    print(f"nums = {nums}, n = {n}: bản sai = {wrong_stop(nums, n)}, đúng = {sol.minPatches(nums, n)}")

nums = [2], n = 1: bản sai = 0, đúng = 1
nums = [1, 2], n = 4: bản sai = 0, đúng = 1
nums = [1, 5, 10], n = 8: bản sai = 2, đúng = 2


Với `[2]`, `n = 1`: `miss = 1`, bản sai kiểm tra `1 < 1` → dừng ngay, trả `0`. Nhưng số `1` không tạo được từ `[2]`. Lỗi này hay xuất hiện khi trộn lẫn hai cách hiểu "đã phủ tới `miss`" (đóng) và "phủ `[1, miss)`" (mở). Hãy **viết rõ ý nghĩa của biến** trong comment rồi suy ra điều kiện dừng.

**Sửa:** `while miss <= n` (tương đương "còn thiếu số `miss` và nó `≤ n`").

### Lỗi 3: viết `for x in nums` rồi quên kiểm tra `miss <= n` khi vá

Một cách viết khác trông "gọn" hơn: duyệt `for x in nums`, trước khi nuốt `x` thì vá cho tới khi `x <= miss`. Nếu quên dừng khi đã phủ đủ `n`, ta sẽ vá thêm chỉ để "với tới" một phần tử khổng lồ không cần thiết.

In [25]:
def wrong_for_loop(nums, n):
    miss, patches = 1, 0
    for x in nums:
        while x > miss:                  # SAI: thiếu điều kiện "and miss <= n"
            miss += miss
            patches += 1
        miss += x
    while miss <= n:
        miss += miss
        patches += 1
    return patches


def fixed_for_loop(nums, n):
    miss, patches = 1, 0
    for x in nums:
        if miss > n:                     # đã phủ đủ: phần tử còn lại không cần xét
            break
        while x > miss and miss <= n:
            miss += miss
            patches += 1
        if miss <= n:
            miss += x
    while miss <= n:
        miss += miss
        patches += 1
    return patches


for nums, n in [([1, 100], 5), ([1, 5, 10], 20), ([1, 2, 31, 33], 2147483647)]:
    print(f"nums = {nums}, n = {n}: bản sai = {wrong_for_loop(nums, n)}, "
          f"bản sửa = {fixed_for_loop(nums, n)}, đúng = {sol.minPatches(nums, n)}")

random.seed(3)
for _ in range(2000):
    nums_t = sorted(random.randint(1, 50) for _ in range(random.randint(1, 6)))
    n_t = random.randint(1, 200)
    assert fixed_for_loop(nums_t, n_t) == sol.minPatches(nums_t, n_t)
print("bản sửa khớp lời giải chuẩn trên 2000 test ngẫu nhiên")

nums = [1, 100], n = 5: bản sai = 6, bản sửa = 2, đúng = 2
nums = [1, 5, 10], n = 20: bản sai = 2, bản sửa = 2, đúng = 2
nums = [1, 2, 31, 33], n = 2147483647: bản sai = 28, bản sửa = 28, đúng = 28
bản sửa khớp lời giải chuẩn trên 2000 test ngẫu nhiên


Với `[1, 100]`, `n = 5`: sau khi nuốt `1`, `miss = 2`. Bản sai cố "với tới" `100` bằng cách vá `2, 4, 8, 16, 32, 64` — 6 lần — trong khi chỉ cần `2, 4` là đã phủ `[1, 8) ⊇ [1, 5]`.

**Sửa:** dùng một vòng `while miss <= n` duy nhất như lời giải chính (khó sai nhất), hoặc thêm điều kiện `miss <= n` ở mọi chỗ như `fixed_for_loop`.

### Lỗi 4: dùng biến "đóng" `reach` nhưng cập nhật theo công thức của `miss`

Nhiều lời giải trên mạng dùng `reach` = số lớn nhất đã phủ, với ý nghĩa *mọi số trong `[1, reach]`* (đóng) tạo được. Quan hệ: `reach = miss − 1`. Khi đổi sang `reach`, **cả ba** chỗ đều phải dịch theo:

| | Biến `miss` (mở) | Biến `reach` (đóng) |
|---|---|---|
| khởi tạo | `miss = 1` | `reach = 0` |
| còn thiếu khi | `miss <= n` | `reach < n` |
| nuốt được khi | `nums[i] <= miss` | `nums[i] <= reach + 1` |
| vá số | `miss` → `miss += miss` | `reach + 1` → `reach += reach + 1` |

Hai lỗi kinh điển: điều kiện nuốt viết `nums[i] <= reach` (thiếu `+ 1`), và vá viết `reach += reach` (với `reach = 0` thì **lặp vô hạn**, vì `0 + 0 = 0`!).

In [26]:
def reach_version(nums, n, bug=None, max_loops=10_000):
    reach, i, patches, loops = 0, 0, 0, 0     # mọi số trong [1, reach] đã tạo được
    while reach < n:
        loops += 1
        if loops > max_loops:
            return "LẶP VÔ HẠN"
        limit = reach if bug == "nuốt" else reach + 1
        if i < len(nums) and nums[i] <= limit:
            reach += nums[i]
            i += 1
        else:
            reach += reach if bug == "vá" else reach + 1
            patches += 1
    return patches


for nums, n in [([1, 2, 2], 5), ([1, 5, 10], 20), ([2], 3)]:
    print(f"nums = {nums}, n = {n}: reach đúng = {reach_version(nums, n)}, "
          f"bug nuốt = {reach_version(nums, n, 'nuốt')}, bug vá = {reach_version(nums, n, 'vá')}, "
          f"chuẩn = {sol.minPatches(nums, n)}")

nums = [1, 2, 2], n = 5: reach đúng = 0, bug nuốt = 1, bug vá = 0, chuẩn = 0
nums = [1, 5, 10], n = 20: reach đúng = 2, bug nuốt = 2, bug vá = 3, chuẩn = 2
nums = [2], n = 3: reach đúng = 1, bug nuốt = 2, bug vá = LẶP VÔ HẠN, chuẩn = 1


### Lỗi 5: tràn số khi viết bằng ngôn ngữ khác, và brute force với `n` lớn

- **Tràn số (overflow):** ở mục 6.3, `miss` cuối cùng là `3 221 225 472 > 2^31 − 1`. Python không sao, nhưng trong Java/C++ nếu khai báo `int miss` thì `miss += miss` sẽ tràn thành số âm, vòng lặp `miss <= n` chạy mãi. Phải dùng `long` / `long long`. Khi dịch lời giải sang ngôn ngữ khác, hãy nhớ điểm này.
- **Brute force / tập tổng với `n` lớn:** gọi `smallest_missing` với `n = 2^31 − 1` sẽ cố tạo một `set` 2 tỉ phần tử → hết RAM. Cách 1 và Cách 2 chỉ dùng để học và kiểm tra trên `n` nhỏ.
- **Quên `nums` có thể chứa số lớn hơn `n`:** lời giải chính tự xử lý đúng (dừng trước khi chạm tới chúng). Nhưng nếu bạn viết kiểu "duyệt hết `nums` trước rồi mới xét `n`", hãy cẩn thận như Lỗi 3.

## 11. Mẹo nhận dạng pattern và biến thể

### 11.1 Dấu hiệu nhận ra pattern "đoạn phủ `[1, miss)`"

- Đề hỏi về **mọi số trong một đoạn liên tiếp** `1..n` tạo được bằng **tổng tập con** (đồng xu, quả cân, mệnh giá…).
- Mỗi phần tử dùng **tối đa một lần** (nếu dùng không giới hạn thì là bài **đồng xu không giới hạn** — khác hẳn, thường là DP).
- Có thể **sắp xếp** dữ liệu (hoặc đề cho sẵn đã sắp xếp), và giá trị `n` / tổng rất lớn khiến DP tập tổng không khả thi.
- Câu hỏi kiểu: "số nhỏ nhất không tạo được", "có bao nhiêu giá trị liên tiếp từ 0 tạo được", "thêm ít nhất bao nhiêu phần tử để phủ".

**Câu thần chú:** *"Sắp xếp tăng dần. Giữ `miss` = số nhỏ nhất chưa tạo được. Gặp `x ≤ miss` thì `miss += x`. Gặp `x > miss` là kẹt: không vá được thì `miss` là đáp án; được vá thì vá chính `miss`."*

### 11.2 Bài liên quan

| Bài | Khác gì so với 330 | Mấu chốt |
|---|---|---|
| Smallest non-constructible value (bài phỏng vấn kinh điển) | không được vá, hỏi số nhỏ nhất không tạo được | sắp xếp + quét, dừng ở `x > miss` |
| LeetCode 1798 — Maximum Number of Consecutive Values You Can Make (Medium) | mảng `coins` chưa sắp xếp, không vá, đếm số giá trị liên tiếp từ `0` | `sort` rồi quét; đáp án chính là `miss` (đếm cả số 0) |
| LeetCode 2952 — Minimum Number of Coins to be Added (Medium) | gần như y hệt 330 nhưng `coins` **chưa sắp xếp** | `sort` trước rồi chạy đúng greedy 330 (bài tập 5) |
| LeetCode 2589 / 1798 nhóm "greedy trên đoạn phủ" | | luôn có một biến mô tả "đã phủ tới đâu" |
| LeetCode 45 — Jump Game II, 1326 — Minimum Number of Taps | phủ đoạn bằng các khoảng | cùng tinh thần "mở rộng vùng phủ xa nhất có thể mỗi bước" |

Cell dưới cài đặt LeetCode 1798 bằng đúng phép quét `miss` (không vá) và thử ví dụ của đề đó; phần sau của cell minh họa biến thể ở mục 11.3.

### 11.3 Hai biến thể nhỏ của chính bài 330

- **Đổi biến `miss` ↔ `reach`**: đã xem ở Lỗi 4. Hai cách tương đương, chọn một và **ghi rõ ý nghĩa** trong comment.
- **Trả về danh sách số vá** thay vì số lượng: chỉ cần `patches.append(miss)` trước khi gấp đôi (`greedy_patch_list` ở mục 7.1). Đây là một bộ vá tối ưu, nhưng không phải bộ duy nhất — ví dụ `[1, 3]`, `n = 6` vá `2` là duy nhất, còn `[1, 5, 10]`, `n = 20` có các bộ khác như `[2, 3]`… hãy tự kiểm tra bằng `covers`.

In [27]:
def get_maximum_consecutive(coins):
    # LeetCode 1798: số giá trị liên tiếp 0, 1, 2, ... tạo được (đếm cả 0).
    miss = 1                         # tạo được [0, miss)  -> có đúng miss giá trị
    for c in sorted(coins):
        if c > miss:
            break
        miss += c
    return miss


assert get_maximum_consecutive([1, 3]) == 2
assert get_maximum_consecutive([1, 1, 1, 4]) == 8
assert get_maximum_consecutive([1, 4, 10, 3, 1]) == 20
print("1798:", get_maximum_consecutive([1, 3]), get_maximum_consecutive([1, 1, 1, 4]),
      get_maximum_consecutive([1, 4, 10, 3, 1]))

print("greedy vá:", greedy_patch_list([1, 5, 10], 20))
for patch in [[2, 4], [2, 3], [2, 2], [1, 3], [3, 4]]:
    print(f"vá {patch}: phủ [1, 20]? {covers([1, 5, 10] + patch, 20)}")

1798: 2 8 20
greedy vá: [2, 4]
vá [2, 4]: phủ [1, 20]? True
vá [2, 3]: phủ [1, 20]? True
vá [2, 2]: phủ [1, 20]? True
vá [1, 3]: phủ [1, 20]? True
vá [3, 4]: phủ [1, 20]? False


## 12. Bài tập luyện tập

Sáu bài, tăng dần độ khó. Hãy tự làm **trước** khi xem gợi ý (mục 13) và đáp án (mục 15). Các cell TODO chỉ định nghĩa hàm, test để sẵn dưới dạng comment — làm xong thì bỏ dấu `#` để tự kiểm tra.

### Bài 1 — Đoán output (không chạy code)

Dự đoán `Solution().minPatches(nums, n)` và **các số được vá** (theo greedy):

1. `nums = [2]`, `n = 8`
2. `nums = [1, 7, 21, 31, 34, 37, 40, 43, 49, 87, 90, 92, 93, 98, 99]`, `n = 12`
3. `nums = [1, 1, 1, 1]`, `n = 4` và `n = 5`
4. `nums = [1, 2, 31, 33]`, `n = 2147483647` (đã gặp ở mục 6.3 — thử tính lại không nhìn)

### Bài 2 — Tính tay bảng trace

Lập bảng trace (giống mục 6.2) cho `nums = [1, 4, 10]`, `n = 50`. Mỗi dòng ghi `miss` trước, `nums[i]`, hành động, `miss` sau, `patches`. Kiểm tra lại ở cuối: `miss = 1 + tổng mọi phần tử đã xét`.

### Bài 3 — Số nhỏ nhất không tạo được (điền TODO)

Viết `smallest_unreachable(nums)` cho `nums` **đã sắp xếp tăng**, **không vá gì**: trả về số nguyên dương nhỏ nhất **không** tạo được bằng tổng tập con. Yêu cầu `O(m)` thời gian, `O(1)` bộ nhớ.

Ví dụ: `[1, 1, 3, 8]` → `6` (tạo được `1..5`; `8 > 6` nên kẹt). `[2, 3]` → `1`. `[1, 2, 4]` → `8`.

In [28]:
def smallest_unreachable(nums):
    # TODO: quét với biến miss, dừng khi gặp phần tử > miss
    raise NotImplementedError


# Tự làm xong rồi mới mở test:
# assert smallest_unreachable([1, 1, 3, 8]) == 6
# assert smallest_unreachable([2, 3]) == 1
# assert smallest_unreachable([1, 2, 4]) == 8
# assert smallest_unreachable([1, 2, 5]) == 4

### Bài 4 — Trả về danh sách số vá (tự viết lại lời giải)

**Không nhìn** code phía trên, viết lại `patch_list(nums, n)` trả về **danh sách các số được vá** theo greedy (danh sách rỗng nếu không cần vá). Viết xong, tự kiểm tra rằng `len(patch_list(...))` khớp `Solution().minPatches(...)` và mảng sau khi vá thật sự phủ `[1, n]` (dùng `covers` với `n` nhỏ).

In [29]:
def patch_list(nums, n):
    # TODO: giống lời giải chính, nhưng ghi lại giá trị miss mỗi lần vá
    raise NotImplementedError


# assert patch_list([1, 3], 6) == [2]
# assert patch_list([1, 5, 10], 20) == [2, 4]
# assert patch_list([1, 2, 2], 5) == []
# assert covers([1, 5, 10] + patch_list([1, 5, 10], 20), 20)

### Bài 5 — Biến thể: LeetCode 2952 *Minimum Number of Coins to be Added*

Cho mảng `coins` (**chưa sắp xếp**) và số nguyên `target`. Một số `x` gọi là *obtainable* nếu tồn tại một dãy con của `coins` có tổng bằng `x`. Trả về số đồng xu **ít nhất** (giá trị tùy ý) cần thêm để **mọi** số trong `[1, target]` đều obtainable.

Ví dụ của đề 2952:

- `coins = [1, 4, 10]`, `target = 19` → `2` (thêm `2` và `8`)
- `coins = [1, 4, 10, 5, 7, 19]`, `target = 19` → `1` (thêm `2`)
- `coins = [1, 1, 1]`, `target = 20` → `3` (thêm `4`, `8`, `16`)

Câu hỏi phụ: vì sao **bắt buộc** phải sắp xếp trước? Cho một phản ví dụ nếu chạy greedy trên mảng chưa sắp xếp.

In [30]:
def minimum_added_coins(coins, target):
    # TODO: sắp xếp rồi áp dụng greedy miss
    raise NotImplementedError


# assert minimum_added_coins([1, 4, 10], 19) == 2
# assert minimum_added_coins([1, 4, 10, 5, 7, 19], 19) == 1
# assert minimum_added_coins([1, 1, 1], 20) == 3

### Bài 6 — Mini challenge: ngân sách `k` lần vá

Bây giờ ta **chỉ được vá tối đa `k` lần**. Hãy tìm số `R` **lớn nhất** sao cho sau khi vá (tối đa `k` lần) mảng phủ được `[1, R]`. (`R = 0` nghĩa là không phủ nổi cả số `1`.)

Ví dụ:

- `nums = [1, 5, 10]`, `k = 0` → `1` (chỉ tạo được `1`, kẹt ở `2`)
- `nums = [1, 5, 10]`, `k = 1` → `3` (vá `2` → phủ `[1, 3]`; muốn nuốt `5` thì phải tạo được cả `4` — một lần vá không đủ)
- `nums = [1, 5, 10]`, `k = 2` → `22`
- `nums = [3]`, `k = 2` → `6`
- `nums = [1, 2, 31, 33]`, `k = 3` → `95`

Bẫy dễ mắc: với `k = 1` nhiều người tính "vá `2` rồi nuốt `5`, `10`" thành `1 + 2 + 5 + 10 = 18`. Sai, vì sau khi vá `2` đoạn phủ chỉ là `[1, 4)`, chưa "đỡ" được số `5`. Đáp án có kèm **kiểm tra với brute force**, vì đây là chỗ dễ "tưởng đúng mà sai".

In [31]:
def max_cover_with_budget(nums, k):
    # TODO: chạy greedy nhưng thay điều kiện dừng "miss > n" bằng "hết ngân sách và bị kẹt"
    raise NotImplementedError


# assert max_cover_with_budget([1, 5, 10], 0) == 1
# assert max_cover_with_budget([1, 5, 10], 1) == 3
# assert max_cover_with_budget([1, 5, 10], 2) == 22
# assert max_cover_with_budget([3], 2) == 6
# assert max_cover_with_budget([1, 2, 31, 33], 3) == 95

## 13. Gợi ý theo mức

**Bài 1**
- Mức 1: bắt đầu `miss = 1`, mỗi bước hỏi "`nums[i] <= miss`?".
- Mức 2: câu 2 — chỉ có vài phần tử đầu là đáng quan tâm, vì vòng lặp dừng ngay khi `miss > 12`.
- Mức 3: câu 3 — bốn số `1` phủ `[1, 5)`; với `n = 4` đã đủ, với `n = 5` thì kẹt ở `miss = 5`.

**Bài 2**
- Mức 1: số `4` nuốt được khi `miss ≥ 4`; số `10` cần `miss ≥ 10`.
- Mức 2: sau khi hết phần tử, chỉ còn gấp đôi `miss` tới khi `> 50`.
- Mức 3: đáp án có 3 lần vá; `miss` cuối cùng là `52`.

**Bài 3**
- Mức 1: đây là Nhận xét 2 (mục 5.2) — không có nhánh vá.
- Mức 2: `for x in nums: if x > miss: break; miss += x`.
- Mức 3: trả về `miss` (cả khi duyệt hết mảng mà không kẹt).

**Bài 4**
- Mức 1: chép khung `while miss <= n` của lời giải chính.
- Mức 2: ở nhánh vá, `append(miss)` **trước** khi `miss += miss` (nếu làm sau, bạn sẽ ghi giá trị đã gấp đôi).
- Mức 3: dùng `covers` để kiểm tra ngẫu nhiên với `n ≤ 30`.

**Bài 5**
- Mức 1: sắp xếp `coins` rồi gọi lại đúng greedy của bài 330.
- Mức 2: phản ví dụ khi không sắp xếp: thử `coins = [4, 1]`, `target = 5` — greedy gặp `4 > 1` sẽ vá `1`, rồi `4 > 2` lại vá `2`… so với đáp án đúng.
- Mức 3: độ phức tạp thành `O(m log m + log target)` vì phải sắp xếp.

**Bài 6**
- Mức 1: chứng minh ở mục 8 vẫn đúng khi không có `n`: greedy làm `miss` lớn nhất sau mỗi số lần vá.
- Mức 2: vòng lặp: nếu `nums[i] <= miss` thì nuốt; ngược lại nếu còn ngân sách thì vá `miss`; ngược lại **dừng**. Trả về `miss − 1`.
- Mức 3: chỉ **dừng** khi đồng thời bị kẹt (`i == len(nums)` hoặc `nums[i] > miss`) **và** đã dùng hết `k` lần vá. Với `[1, 5, 10]`, `k = 2`: nuốt `1` → vá `2` → vá `4` → nuốt `5` → nuốt `10` → `miss = 23`, hết phần tử và hết ngân sách → `R = 22`.

## 14. Template cuối cùng nên nhớ

```python
class Solution:
    def minPatches(self, nums: List[int], n: int) -> int:
        miss, i, patches = 1, 0, 0          # mọi số trong [1, miss) đã tạo được
        while miss <= n:                    # còn thiếu số miss (<= n)
            if i < len(nums) and nums[i] <= miss:
                miss += nums[i]             # nuốt miễn phí
                i += 1
            else:
                miss += miss                # vá chính miss -> phủ gấp đôi
                patches += 1
        return patches
```

**Câu thần chú**: *"`miss` là số nhỏ nhất chưa tạo được. Có `nums[i] ≤ miss` thì nuốt, `miss += nums[i]`. Không có thì vá `miss`, `miss` gấp đôi. Dừng khi `miss > n`."*

Vì sao tin được: `miss = 1 + tổng các số đã xét`; phần tử `> miss` không bao giờ giúp tạo `miss`; trong mọi số vá `≤ miss`, vá `miss` cho đoạn phủ dài nhất, mà đoạn phủ dài hơn không bao giờ thiệt.

## 15. Checklist tự kiểm tra trước khi submit

- Biến `miss` khởi tạo bằng `1` (không phải `0`) và có comment ý nghĩa "`[1, miss)` đã phủ"?
- Điều kiện lặp là `miss <= n`, **không** phải `miss < n`?
- Điều kiện nuốt là `nums[i] <= miss` (có dấu `=`) và kiểm tra `i < len(nums)` **trước** khi đọc `nums[i]`?
- Ở nhánh vá: **không** tăng `i` (phần tử `nums[i]` vẫn còn đó để nuốt ở vòng sau)?
- Nếu đổi sang biến `reach`: đã dịch đủ ba chỗ (`reach = 0`, `reach < n`, `nums[i] <= reach + 1`, `reach += reach + 1`)?
- Nếu dịch sang Java/C++: `miss` là `long` để không tràn khi gấp đôi gần `2^31`?
- Đã test: 3 ví dụ của đề, `n = 1` với `nums = [2]`, `nums` toàn số `> n`, `nums` có số trùng, `[1, 2, 31, 33]` với `n = 2^31 − 1` (đáp án `28`)?

## 16. Đáp án tham khảo

### Đáp án Bài 1

1. `[2]`, `n = 8` → **3** lần vá: `1` (miss `1 → 2`), nuốt `2` (`→ 4`), vá `4` (`→ 8`), vá `8` (`→ 16 > 8`). Các số vá: `[1, 4, 8]`.
2. **2** lần vá: nuốt `1` (`→ 2`), `7 > 2` → vá `2` (`→ 4`), `7 > 4` → vá `4` (`→ 8`), nuốt `7` (`→ 15 > 12`) dừng. 11 phần tử phía sau không bao giờ được xét.
3. Bốn số `1` phủ `[1, 5)`: với `n = 4` → **0**; với `n = 5` → **1** (vá `5`, phủ `[1, 10)`).
4. **28**: vá `4, 8, 16` rồi 25 lần gấp đôi từ `96` (mục 6.3).

### Đáp án Bài 2 — `nums = [1, 4, 10]`, `n = 50`

| Bước | `miss` trước | `nums[i]` | Hành động | `miss` sau | `patches` |
|---:|---:|---:|---|---:|---:|
| 1 | 1 | 1 | nuốt 1 | 2 | 0 |
| 2 | 2 | 4 | `4 > 2` → **vá 2** | 4 | 1 |
| 3 | 4 | 4 | nuốt 4 (`4 <= 4`) | 8 | 1 |
| 4 | 8 | 10 | `10 > 8` → **vá 8** | 16 | 2 |
| 5 | 16 | 10 | nuốt 10 | 26 | 2 |
| 6 | 26 | hết | **vá 26** | 52 | 3 |

Dừng vì `52 > 50`. Đáp án **3** (vá `2, 8, 26`). Kiểm tra: `1 + (1 + 2 + 4 + 8 + 10 + 26) = 52` ✔. Bước 3 là chỗ dễ sai nếu dùng `<` thay `<=` (Lỗi 1).

In [32]:
for nums, n in [([2], 8),
                ([1, 7, 21, 31, 34, 37, 40, 43, 49, 87, 90, 92, 93, 98, 99], 12),
                ([1, 1, 1, 1], 4), ([1, 1, 1, 1], 5)]:
    print(f"n = {n:2}: minPatches = {sol.minPatches(nums, n)}, vá {greedy_patch_list(nums, n)}")
p4 = greedy_patch_list([1, 2, 31, 33], 2147483647)
print("câu 4:", len(p4), "lần vá, 5 số đầu:", p4[:5])
assert [sol.minPatches([2], 8), sol.minPatches([1, 1, 1, 1], 4), sol.minPatches([1, 1, 1, 1], 5), len(p4)] == [3, 0, 1, 28]

assert trace_greedy([1, 4, 10], 50) == 3
assert 1 + sum([1, 4, 10] + greedy_patch_list([1, 4, 10], 50)) == 52

n =  8: minPatches = 3, vá [1, 4, 8]
n = 12: minPatches = 2, vá [2, 4]
n =  4: minPatches = 0, vá []
n =  5: minPatches = 1, vá [5]
câu 4: 28 lần vá, 5 số đầu: [4, 8, 16, 96, 192]
bước |       miss | nums[i] | hành động      |   miss sau | patches
   1 |          1 |       1 | nuốt 1         |          2 | 0
   2 |          2 |       4 | VÁ 2           |          4 | 1
   3 |          4 |       4 | nuốt 4         |          8 | 1
   4 |          8 |      10 | VÁ 8           |         16 | 2
   5 |         16 |      10 | nuốt 10        |         26 | 2
   6 |         26 |     hết | VÁ 26          |         52 | 3
dừng: miss = 52 > n = 50  ->  đáp án 3


### Đáp án Bài 3 và Bài 4

Bài 3 chính là Nhận xét 2: quét tăng dần, kẹt ở phần tử đầu tiên `> miss`. Bài 4 là lời giải chính, ghi lại `miss` trước khi gấp đôi. Kiểm tra Bài 3 với `contiguous_miss` (tập tổng) và Bài 4 với `covers` trên dữ liệu ngẫu nhiên.

In [33]:
def smallest_unreachable_solution(nums):
    miss = 1
    for x in nums:
        if x > miss:
            break
        miss += x
    return miss


def patch_list_solution(nums, n):
    miss, i, patches = 1, 0, []
    while miss <= n:
        if i < len(nums) and nums[i] <= miss:
            miss += nums[i]
            i += 1
        else:
            patches.append(miss)         # ghi lại TRƯỚC khi gấp đôi
            miss += miss
    return patches


assert smallest_unreachable_solution([1, 1, 3, 8]) == 6
assert smallest_unreachable_solution([2, 3]) == 1
assert smallest_unreachable_solution([1, 2, 4]) == 8
assert smallest_unreachable_solution([1, 2, 5]) == 4
assert patch_list_solution([1, 3], 6) == [2]
assert patch_list_solution([1, 5, 10], 20) == [2, 4]
assert patch_list_solution([1, 2, 2], 5) == []

random.seed(34)
for _ in range(300):
    nums_t = sorted(random.randint(1, 15) for _ in range(random.randint(1, 6)))
    n_t = random.randint(1, 30)
    assert smallest_unreachable_solution(nums_t) == contiguous_miss(nums_t)
    p = patch_list_solution(nums_t, n_t)
    assert covers(nums_t + p, n_t) and len(p) == sol.minPatches(nums_t, n_t)
print("Bài 3 và Bài 4: đúng trên ví dụ và 300 test ngẫu nhiên")

Bài 3 và Bài 4: đúng trên ví dụ và 300 test ngẫu nhiên


### Đáp án Bài 5 — LeetCode 2952

Sắp xếp `coins` rồi chạy greedy y hệt bài 330 (thay `n` bằng `target`). Thời gian `O(m log m + log target)`.

**Vì sao phải sắp xếp?** Nhận xét 2 cần "mọi phần tử chưa xét đều `≥` phần tử hiện tại". Mảng chưa sắp xếp phá vỡ điều đó: với `coins = [4, 1]`, `target = 5`, greedy không sắp xếp gặp `4 > 1` nên vá `1`, gặp `4 > 2` lại vá `2`, rồi nuốt `4` (`→ 8`)… tổng cộng **2** lần vá, trong khi đáp án đúng là **1**: có sẵn `1` và `4` nên phủ `1, 4, 5`, chỉ cần vá `2` (→ `[1, 2, 4]` phủ `[1, 8)`).

In [34]:
def minimum_added_coins_solution(coins, target):
    coins = sorted(coins)
    miss, i, added = 1, 0, 0
    while miss <= target:
        if i < len(coins) and coins[i] <= miss:
            miss += coins[i]
            i += 1
        else:
            miss += miss
            added += 1
    return added


def unsorted_greedy(coins, target):          # SAI: không sắp xếp
    return Solution().minPatches(coins, target)


assert minimum_added_coins_solution([1, 4, 10], 19) == 2
assert minimum_added_coins_solution([1, 4, 10, 5, 7, 19], 19) == 1
assert minimum_added_coins_solution([1, 1, 1], 20) == 3
print("[4, 1], target 5: không sắp xếp =", unsorted_greedy([4, 1], 5),
      "| có sắp xếp =", minimum_added_coins_solution([4, 1], 5),
      "| trọng tài =", min_patches_brute([4, 1], 5))

[4, 1], target 5: không sắp xếp = 2 | có sắp xếp = 1 | trọng tài = 1


### Đáp án Bài 6 — Ngân sách `k` lần vá

**Lời giải:** chạy greedy không có `n`; khi bị kẹt (`nums[i] > miss` hoặc hết phần tử) mà còn ngân sách thì vá `miss`, hết ngân sách thì dừng. Trả về `miss − 1`.

Vì sao đúng: lập luận "greedy luôn đi trước" ở mục 8.2 không dùng `n` ở đâu cả — nó nói rằng sau mỗi số lần vá `p`, `miss` của greedy lớn nhất trong mọi cách vá. Vậy với ngân sách `k`, `miss` của greedy sau `k` lần vá (và nuốt hết mọi thứ có thể) là lớn nhất có thể.

Còn bẫy "`18`" với `[1, 5, 10]`, `k = 1`: greedy kẹt ở `miss = 2`, vá `2` → `miss = 4`; vòng tiếp theo `5 > 4` mà hết ngân sách → dừng, `R = 3`. Có số vá nào khác tốt hơn không? Vá `y = 1` → `[1, 3)`; vá `y ≥ 3` thì số `2` không bao giờ tạo được → `R = 1`. Không cách nào nuốt được `5` với một lần vá, nên `18` là không thể. Brute force ở cell dưới xác nhận điều này.

Hàm brute force thử mọi bộ `≤ k` số vá. Giá trị vá được chặn trên bởi `(1 + sum(nums)) · 2^k`: một số vá lớn hơn `miss` tại vị trí của nó không giúp được gì, mà `miss` không bao giờ vượt quá `1 + tổng mọi thứ`, và mỗi lần vá tổng tăng tối đa gấp đôi.

In [35]:
def max_cover_with_budget_solution(nums, k):
    miss, i, used = 1, 0, 0
    while True:
        if i < len(nums) and nums[i] <= miss:
            miss += nums[i]
            i += 1
        elif used < k:
            miss += miss
            used += 1
        else:
            return miss - 1


def max_cover_brute(nums, k):
    # thử mọi bộ <= k số vá; giá trị vá có ích không vượt quá (1 + tổng nums) * 2^k
    cap = (1 + sum(nums)) * 2 ** k
    best = 0
    for j in range(k + 1):
        for patch in combinations_with_replacement(range(1, cap + 1), j):
            best = max(best, scan_miss(list(nums) + list(patch)) - 1)
    return best


for nums, k in [([1, 5, 10], 0), ([1, 5, 10], 1), ([1, 5, 10], 2), ([3], 1), ([3], 2)]:
    print(f"nums = {nums}, k = {k}: greedy = {max_cover_with_budget_solution(nums, k)}, "
          f"brute = {max_cover_brute(nums, k)}")

assert max_cover_with_budget_solution([1, 5, 10], 0) == 1
assert max_cover_with_budget_solution([1, 5, 10], 1) == 3        # không phải 18!
assert max_cover_with_budget_solution([1, 5, 10], 2) == 22
assert max_cover_with_budget_solution([3], 2) == 6               # vá 1, 2 -> phủ [1, 3] rồi nuốt 3 -> [1, 6]
assert max_cover_with_budget_solution([1, 2, 31, 33], 3) == 95

random.seed(6)
for _ in range(80):
    nums_t = sorted(random.randint(1, 6) for _ in range(random.randint(1, 3)))
    k_t = random.randint(0, 2)
    assert max_cover_with_budget_solution(nums_t, k_t) == max_cover_brute(nums_t, k_t), (nums_t, k_t)
print("Bài 6: greedy khớp brute force trên 80 test ngẫu nhiên")

nums = [1, 5, 10], k = 0: greedy = 1, brute = 1
nums = [1, 5, 10], k = 1: greedy = 3, brute = 3
nums = [1, 5, 10], k = 2: greedy = 22, brute = 22
nums = [3], k = 1: greedy = 1, brute = 1
nums = [3], k = 2: greedy = 6, brute = 6
Bài 6: greedy khớp brute force trên 80 test ngẫu nhiên


Greedy và brute force **khớp nhau** ở mọi dòng. Bài học rút ra: *đừng tin một con số chỉ vì nó "nghe hợp lý" (như `18`) — hãy kiểm tra bằng brute force*.

Cell trên cũng chạy các assert của Bài 6 và đối chiếu ngẫu nhiên greedy với brute force trên input nhỏ. (Ví dụ `[1, 2, 31, 33]`, `k = 3` có giá trị vá tới hàng trăm nên brute force quá chậm; ta chỉ kiểm tra bằng tay: vá `4, 8, 16` như mục 6.3 rồi nuốt `31, 33` → `miss = 96` → `R = 95`.)

## Lời kết

Bài 330 là ví dụ điển hình của một lời giải Hard **ngắn nhưng sâu**:

1. Bắt đầu từ định nghĩa (Cách 1: thử mọi bộ vá) — chậm nhưng chắc chắn đúng, làm trọng tài.
2. Đặt câu hỏi "lỗ hổng nhỏ nhất phải được lấp thế nào?" (Cách 2) — ra chiến lược đúng nhưng lưu trữ quá nặng.
3. Nhìn kỹ dữ liệu để thấy **chỉ một con số `miss`** quyết định mọi thứ (Nhận xét 1, 2) — nén `O(n)` bộ nhớ thành `O(1)`, và mỗi lần vá **gấp đôi** đoạn phủ (Cách 3).
4. Chứng minh bằng "greedy luôn đi trước" và kiểm chứng bằng phản ví dụ các chiến lược khác (Cách 4).

Thói quen đáng giữ lại cho mọi bài greedy: **viết brute force làm trọng tài, đối chiếu ngẫu nhiên, và đừng tin một con số chỉ vì nó nghe hợp lý.**